In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
transactions = pd.read_csv('transactions.csv')
store = pd.read_csv('stores.csv')
holidays = pd.read_csv('holidays_events.csv')
oil = pd.read_csv('oil.csv')

print(train.info())
train

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB
None


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.000,0
1,1,2013-01-01,1,BABY CARE,0.000,0
2,2,2013-01-01,1,BEAUTY,0.000,0
3,3,2013-01-01,1,BEVERAGES,0.000,0
4,4,2013-01-01,1,BOOKS,0.000,0
...,...,...,...,...,...,...
3000883,3000883,2017-08-15,9,POULTRY,438.133,0
3000884,3000884,2017-08-15,9,PREPARED FOODS,154.553,1
3000885,3000885,2017-08-15,9,PRODUCE,2419.729,148
3000886,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8


In [3]:
print(test.info())
test

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           28512 non-null  int64 
 1   date         28512 non-null  object
 2   store_nbr    28512 non-null  int64 
 3   family       28512 non-null  object
 4   onpromotion  28512 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 1.1+ MB
None


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0
...,...,...,...,...,...
28507,3029395,2017-08-31,9,POULTRY,1
28508,3029396,2017-08-31,9,PREPARED FOODS,0
28509,3029397,2017-08-31,9,PRODUCE,1
28510,3029398,2017-08-31,9,SCHOOL AND OFFICE SUPPLIES,9


In [4]:
train.isnull().sum()

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

In [5]:
print(transactions.info())
transactions

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB
None


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
...,...,...,...
83483,2017-08-15,50,2804
83484,2017-08-15,51,1573
83485,2017-08-15,52,2255
83486,2017-08-15,53,932


In [6]:
print(store.info())
store

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB
None


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4
5,6,Quito,Pichincha,D,13
6,7,Quito,Pichincha,D,8
7,8,Quito,Pichincha,D,8
8,9,Quito,Pichincha,B,6
9,10,Quito,Pichincha,C,15


In [7]:
# city and state both indicate location but city is more specific, so the state column is not needed
store = store.drop(columns=['state'])

In [8]:
print(holidays.info())
holidays

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB
None


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
...,...,...,...,...,...,...
345,2017-12-22,Additional,National,Ecuador,Navidad-3,False
346,2017-12-23,Additional,National,Ecuador,Navidad-2,False
347,2017-12-24,Additional,National,Ecuador,Navidad-1,False
348,2017-12-25,Holiday,National,Ecuador,Navidad,False


In [9]:
holidays.rename(columns={'type': 'holiday_type'}, inplace=True)
print(holidays['holiday_type'].value_counts(), '\n')

holiday_type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64 



In [10]:
transferred = holidays[holidays['transferred'] == True]
transferred

,date,holiday_type,locale,locale_name,description,transferred
19,2012-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
72,2013-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
135,2014-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
255,2016-05-24,Holiday,National,Ecuador,Batalla de Pichincha,True
266,2016-07-25,Holiday,Local,Guayaquil,Fundacion de Guayaquil,True
268,2016-08-10,Holiday,National,Ecuador,Primer Grito de Independencia,True
297,2017-01-01,Holiday,National,Ecuador,Primer dia del ano,True
303,2017-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,True
312,2017-05-24,Holiday,National,Ecuador,Batalla de Pichincha,True
324,2017-08-10,Holiday,National,Ecuador,Primer Grito de Independencia,True


In [11]:
# Since a transferred day is more like a normal day than a holiday, I will drop these days
holidays = holidays.drop(transferred.index)
holidays = holidays.drop(columns=['locale', 'description', 'transferred','locale_name'])

In [12]:
print(oil.info())
oil

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB
None


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
...,...,...
1213,2017-08-25,47.65
1214,2017-08-28,46.40
1215,2017-08-29,46.46
1216,2017-08-30,45.96


In [13]:
# fill missing values with the next value
oil.bfill(inplace=True)

In [14]:
# merge dataframe for train data
train = train.drop(columns='id')
train = train.merge(store)
train = train.merge(oil, how='left')
train = train.merge(holidays,how='left')

print(train.info())
train.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3054348 entries, 0 to 3054347
Data columns (total 10 columns):
 #   Column        Dtype  
---  ------        -----  
 0   date          object 
 1   store_nbr     int64  
 2   family        object 
 3   sales         float64
 4   onpromotion   int64  
 5   city          object 
 6   type          object 
 7   cluster       int64  
 8   dcoilwtico    float64
 9   holiday_type  object 
dtypes: float64(2), int64(3), object(5)
memory usage: 233.0+ MB
None


,date,store_nbr,family,sales,onpromotion,city,type,cluster,dcoilwtico,holiday_type
0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,D,13,93.14,Holiday
1,2013-01-01,1,BABY CARE,0.0,0,Quito,D,13,93.14,Holiday
2,2013-01-01,1,BEAUTY,0.0,0,Quito,D,13,93.14,Holiday
3,2013-01-01,1,BEVERAGES,0.0,0,Quito,D,13,93.14,Holiday
4,2013-01-01,1,BOOKS,0.0,0,Quito,D,13,93.14,Holiday


In [15]:
# split column date to 3 columns: year, month, day
train['date'] = pd.to_datetime(train['date'])
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day_of_week
train = train.drop(columns='date')

# label encode categorical data
label_encode = LabelEncoder()
train['family'] = label_encode.fit_transform(train['family'])
train['city'] = label_encode.fit_transform(train['city'])
train['type'] = label_encode.fit_transform(train['type'])
train['holiday_type'] = label_encode.fit_transform(train['holiday_type'])

print(train.info())
train

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3054348 entries, 0 to 3054347
Data columns (total 11 columns):
 #   Column        Dtype  
---  ------        -----  
 0   store_nbr     int64  
 1   family        int32  
 2   sales         float64
 3   onpromotion   int64  
 4   city          int32  
 5   type          int32  
 6   cluster       int64  
 7   dcoilwtico    float64
 8   holiday_type  int32  
 9   month         int32  
 10  day           int32  
dtypes: float64(2), int32(6), int64(3)
memory usage: 186.4 MB
None


,store_nbr,family,sales,onpromotion,city,type,cluster,dcoilwtico,holiday_type,month,day
0,1,0,0.000,0,18,3,13,93.14,3,1,1
1,1,1,0.000,0,18,3,13,93.14,3,1,1
2,1,2,0.000,0,18,3,13,93.14,3,1,1
3,1,3,0.000,0,18,3,13,93.14,3,1,1
4,1,4,0.000,0,18,3,13,93.14,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...
3054343,9,28,438.133,0,18,1,6,47.57,3,8,1
3054344,9,29,154.553,1,18,1,6,47.57,3,8,1
3054345,9,30,2419.729,148,18,1,6,47.57,3,8,1
3054346,9,31,121.000,8,18,1,6,47.57,3,8,1


In [ ]:
x_train = train.drop(columns = 'sales')
#[['store_nbr', 'family', 'onpromotion', 'city', 'dcoilwtico', 'day', 'month']]
y_train = train['sales']

xgbr = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=0)
xgbr.fit(x_train,y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=0, ...)

In [17]:
df = pd.DataFrame(test['id'])
# merge dataframe test data
test = test.drop(columns='id')
test = test.merge(store)
test = test.merge(oil, how='left')
test = test.merge(holidays,how='left')

# split column date to 3 columns: year, month, day
test['date'] = pd.to_datetime(test['date'])
test['month'] = test['date'].dt.month
test['day'] = test['date'].dt.day_of_week
test = test.drop(columns='date')

# label encode categorical data
test['family'] = label_encode.fit_transform(test['family'])
test['city'] = label_encode.fit_transform(test['city'])
test['type'] = label_encode.fit_transform(test['type'])
test['holiday_type'] = label_encode.fit_transform(test['holiday_type'])

print(test.info())
test

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   store_nbr     28512 non-null  int64  
 1   family        28512 non-null  int32  
 2   onpromotion   28512 non-null  int64  
 3   city          28512 non-null  int32  
 4   type          28512 non-null  int32  
 5   cluster       28512 non-null  int64  
 6   dcoilwtico    21384 non-null  float64
 7   holiday_type  28512 non-null  int32  
 8   month         28512 non-null  int32  
 9   day           28512 non-null  int32  
dtypes: float64(1), int32(6), int64(3)
memory usage: 1.5 MB
None


,store_nbr,family,onpromotion,city,type,cluster,dcoilwtico,holiday_type,month,day
0,1,0,0,18,3,13,46.80,1,8,2
1,1,1,0,18,3,13,46.80,1,8,2
2,1,2,2,18,3,13,46.80,1,8,2
3,1,3,20,18,3,13,46.80,1,8,2
4,1,4,0,18,3,13,46.80,1,8,2
...,...,...,...,...,...,...,...,...,...,...
28507,9,28,1,18,1,6,47.26,1,8,3
28508,9,29,0,18,1,6,47.26,1,8,3
28509,9,30,1,18,1,6,47.26,1,8,3
28510,9,31,9,18,1,6,47.26,1,8,3


In [18]:
df['sales'] = xgbr.predict(test)
df.to_csv('store_sales_time_series_forecasting.csv', index=False)
df

,id,sales
0,3000888,3.320786
1,3000889,1.545011
2,3000890,-31.116352
3,3000891,2350.536621
4,3000892,-24.200905
...,...,...
28507,3029395,399.315643
28508,3029396,203.076630
28509,3029397,1589.583740
28510,3029398,742.123535
